# MVP de Engenharia de Dados — Etapa 2: Coleta**Tema:** ocorrências criminais registradas no município de Sorocaba (SP)**Plataforma:** Google Cloud Platform---Esta etapa cumpre o item **2. Coleta** do trabalho: *"uma vez definido o conjunto de dados, devemos coletar e armazená-los na nuvem"*.O destino da coleta é o **data lake** implementado no Cloud Storage. Seguindo a definição da Aula 3 da disciplina de Data Warehouse e Data Lakes — *"em um data lake, os dados são armazenados em seu formato nativo, ou seja, exatamente como foram gerados"* — o bucket tem duas zonas:| Zona | Conteúdo | Por quê ||---|---|---|| `bruta/` | os arquivos `.xlsx` e `.json` **exatamente como publicados pela fonte** | preserva o dado original, permitindo reprocessar tudo do zero e auditar qualquer transformação || `preparada/` | o mesmo conteúdo convertido para **Parquet**, sem perda | o Spark não lê `.xlsx` nativamente; o Parquet é colunar, comprimido e lido de forma distribuída pelo cluster |Nenhum registro e nenhuma coluna são descartados entre as duas zonas: a conversão é apenas de **formato**. Toda limpeza e conformação acontece na etapa seguinte (ETL), de forma documentada e auditável.### Fontes utilizadas| Fonte | Conteúdo | Licença / condição de uso ||---|---|---|| **SSP-SP — SPDados** (`ssp.sp.gov.br`) | ocorrências criminais registradas no estado de São Paulo, um arquivo `.xlsx` por ano | dados abertos publicados oficialmente pela Secretaria da Segurança Pública para download direto; **não há scraping de páginas** — apenas requisição aos arquivos publicados, com *User-Agent* identificado || **IBGE — API de Localidades** | código, nome e hierarquia regional do município | dados abertos || **IBGE — SIDRA** (tabelas 4709 e 6579) | população de Sorocaba por ano | dados abertos |Os dados do IBGE cumprem o papel de **dados de referência** descrito na Aula 2 de Governança de Dados: *"são utilizados para classificar ou categorizar outros dados... a utilização de referências externas é interessante e deve ser adotada quando for possível, pois permite a comparação e a utilização de dados de várias fontes"*. Aqui eles permitem (a) validar o município e (b) calcular a taxa por 100 mil habitantes.

## 0. Parâmetros e autenticação

In [ ]:
# --- Parâmetros do ambiente (ajuste PROJETO_ID para o seu projeto) --------
PROJETO_ID = "mvp-criminalidade-sorocaba"
REGIAO     = "southamerica-east1"
BUCKET     = f"{PROJETO_ID}-datalake"

# --- Parâmetros da coleta ------------------------------------------------
ANOS      = [2022, 2023, 2024, 2025, 2026]
URL_BASE  = "https://www.ssp.sp.gov.br/assets/estatistica/transparencia/spDados"
ARQUIVO   = "SPDadosCriminais_{ano}.xlsx"
DIR_LOCAL = "/content/dados"

# Código IBGE de Sorocaba: usado apenas para conferência nesta etapa;
# o filtro propriamente dito acontece no ETL.
COD_IBGE_SOROCABA = "3552205"

import os
os.makedirs(DIR_LOCAL, exist_ok=True)
print("Projeto:", PROJETO_ID, "| Região:", REGIAO, "| Bucket:", BUCKET)

In [ ]:
# Autenticação do Colab na GCP
from google.colab import auth
auth.authenticate_user()

!gcloud config set project {PROJETO_ID} --quiet

from google.cloud import storage
cliente_gcs = storage.Client(project=PROJETO_ID)
bucket = cliente_gcs.bucket(BUCKET)
print("Bucket acessível:", bucket.exists())

## 1. Coleta dos arquivos da fonte para a zona brutaO download é feito em blocos e gravado direto em disco (o arquivo de cada ano tem cerca de **200 MB**, e carregá-lo inteiro em memória seria desnecessário). Em seguida o arquivo sobe para a zona bruta do data lake, sem qualquer alteração.A célula é **idempotente**: arquivos que já existem no destino são pulados, de modo que o notebook pode ser reexecutado sem baixar tudo de novo.

In [ ]:
import urllib.request
import datetime as dt

CABECALHO_HTTP = {"User-Agent": "mvp-engenharia-dados-puc/1.0 (trabalho academico)"}


def baixar_da_fonte(ano: int) -> str:
    """Baixa o arquivo anual da SSP-SP para o disco local, em blocos."""
    nome = ARQUIVO.format(ano=ano)
    destino = os.path.join(DIR_LOCAL, nome)
    if os.path.exists(destino):
        print(f"[{ano}] já em disco ({os.path.getsize(destino)/1e6:.0f} MB)")
        return destino

    url = f"{URL_BASE}/{nome}"
    print(f"[{ano}] baixando {url} ...")
    requisicao = urllib.request.Request(url, headers=CABECALHO_HTTP)
    with urllib.request.urlopen(requisicao, timeout=600) as resposta, open(destino, "wb") as saida:
        while True:
            bloco = resposta.read(8 * 1024 * 1024)
            if not bloco:
                break
            saida.write(bloco)
    print(f"[{ano}] {os.path.getsize(destino)/1e6:.0f} MB gravados")
    return destino


def enviar_para_zona_bruta(caminho_local: str, prefixo: str) -> str:
    """Envia o arquivo para a zona bruta do data lake, sem alterá-lo."""
    nome = os.path.basename(caminho_local)
    destino = f"bruta/{prefixo}/{nome}"
    blob = bucket.blob(destino)
    if blob.exists():
        print(f"    já na zona bruta: gs://{BUCKET}/{destino}")
        return destino
    blob.upload_from_filename(caminho_local, timeout=1800)
    print(f"    enviado: gs://{BUCKET}/{destino}")
    return destino


inicio = dt.datetime.now()
for ano in ANOS:
    caminho = baixar_da_fonte(ano)
    enviar_para_zona_bruta(caminho, "ssp-sp")
print("\\nColeta concluída em", dt.datetime.now() - inicio)

## 2. Descoberta do esquemaAntes de modelar qualquer coisa, é preciso saber **o que exatamente** a fonte entrega. Duas informações são levantadas aqui:1. **O dicionário de campos publicado pela própria SSP-SP**, que vem como uma aba dentro de cada arquivo. Ele é a fonte dos *metadados semânticos* do Catálogo de Dados (Aula 1: *"metadados semânticos, que atentam para o significado dos conceitos manipulados"*).2. **Os cabeçalhos reais de cada ano**, porque os nomes de coluna **mudam entre anos**. Essa heterogeneidade é justamente o que o processo de ETL existe para resolver: *"diversas questões de integração (padronização, heterogeneidade semântica) e tratamento da qualidade dos dados são tratadas durante a execução desse processo"* (Aula 1).Nada aqui é suposto: o de-para usado no ETL é construído a partir do que estas células imprimem.

In [ ]:
import unicodedata
import openpyxl
import pandas as pd


def normalizar_texto(valor) -> str:
    """Maiúsculas, sem acentos e sem espaços nas pontas.

    Usada tanto para comparar nomes de coluna quanto valores categóricos,
    já que a fonte varia a grafia entre anos (ex.: CIRCUNSCRIÇÃO x CIRCUNSCRICAO).
    """
    if valor is None:
        return ""
    decomposto = unicodedata.normalize("NFKD", str(valor))
    sem_acento = "".join(c for c in decomposto if not unicodedata.combining(c))
    return sem_acento.strip().upper()


def normalizar_nome_coluna(nome) -> str:
    """Nome canônico de coluna: sem acento, maiúsculo, espaços viram underscore."""
    return normalizar_texto(nome).replace(" ", "_")


def abrir_planilha(ano: int):
    caminho = os.path.join(DIR_LOCAL, ARQUIVO.format(ano=ano))
    return openpyxl.load_workbook(caminho, read_only=True, data_only=True)


def classificar_guias(livro):
    """Separa a aba de dicionário das abas de dados, sem depender do nome exato.

    Necessário porque a fonte já usou 'CAMPOS_DA_TABELA_SPDADOS' (2022) e
    'Campos da Tabela_SPDADOS' (2026) para a mesma aba.
    """
    dicionario, dados = None, []
    for guia in livro.sheetnames:
        if "CAMPO" in normalizar_texto(guia):
            dicionario = guia
        else:
            dados.append(guia)
    return dicionario, dados


def ler_cabecalho(livro, guia):
    linhas = livro[guia].iter_rows(values_only=True)
    return [str(c).strip() for c in next(linhas) if c is not None]


estrutura = {}
for ano in ANOS:
    livro = abrir_planilha(ano)
    guia_dicionario, guias_dados = classificar_guias(livro)
    estrutura[ano] = {
        "dicionario": guia_dicionario,
        "guias": guias_dados,
        "cabecalho": ler_cabecalho(livro, guias_dados[0]),
    }
    livro.close()
    print(f"[{ano}] dicionário: {guia_dicionario!r} | guias de dados: {guias_dados} "
          f"| {len(estrutura[ano]['cabecalho'])} colunas")

In [ ]:
# --- Dicionário de campos publicado pela fonte (metadados semânticos) -----
livro = abrir_planilha(ANOS[-1])
guia_dicionario, _ = classificar_guias(livro)
linhas_dicionario = [l for l in livro[guia_dicionario].iter_rows(values_only=True)
                     if l and l[0] and l[0] != "Campos"]
livro.close()

dicionario_fonte = pd.DataFrame(
    [(str(l[0]).strip(), str(l[1]).strip() if len(l) > 1 and l[1] else "")
     for l in linhas_dicionario if str(l[0]).strip().isupper() or "_" in str(l[0])],
    columns=["campo", "descricao_da_fonte"],
)
print(f"{len(dicionario_fonte)} campos documentados pela SSP-SP:\\n")
pd.set_option("display.max_colwidth", 120)
display(dicionario_fonte)

In [ ]:
# --- De-para real dos cabeçalhos entre anos ------------------------------
# Cada linha é um nome canônico (normalizado); cada coluna mostra como aquele
# campo aparece no arquivo daquele ano, ou '—' quando o campo não existe.
mapa = {}
for ano, info in estrutura.items():
    for coluna in info["cabecalho"]:
        mapa.setdefault(normalizar_nome_coluna(coluna), {})[ano] = coluna

comparativo = pd.DataFrame(
    [{"coluna_canonica": canonica, **{ano: nomes.get(ano, "—") for ano in ANOS}}
     for canonica, nomes in mapa.items()]
).set_index("coluna_canonica")

divergentes = comparativo[comparativo.apply(
    lambda l: len({v for v in l if v != "—"}) > 1 or "—" in list(l), axis=1)]

print(f"{len(comparativo)} colunas distintas no total; "
      f"{len(divergentes)} divergem entre anos (grafia diferente ou campo ausente):\\n")
display(divergentes)

### Consequência para o ETLAs divergências acima são a razão pela qual a etapa de transformação existe. Elas são resolvidas no job Spark por um **de-para explícito**, em que cada campo canônico do data warehouse aceita todas as grafias já usadas pela fonte. O de-para está versionado em [`spark/etl_ocorrencias.py`](../spark/etl_ocorrencias.py) e é aplicado sobre os nomes **normalizados**, de modo que uma nova variação de acento ou de caixa em anos futuros não quebra o pipeline.

## 3. Verificação de dados pessoais (LGPD)A Aula 3 de Governança trata da Lei Geral de Proteção de Dados e recomenda que, *"especialmente quando se utilizam dados pessoais para análises, é importante que eles sejam anonimizados"*. Antes de carregar qualquer coisa no data warehouse, verificamos que tipo de dado a base contém.A célula abaixo levanta as colunas com potencial de identificar pessoas e evidencia o tratamento que a **própria fonte** já aplica.

In [ ]:
# Colunas com potencial de identificação (endereço e geolocalização do fato).
# A base não traz nome, documento, idade ou sexo de vítimas ou autores.
colunas_sensiveis = ["LOGRADOURO", "NUMERO_LOGRADOURO", "LATITUDE", "LONGITUDE", "BAIRRO"]

livro = abrir_planilha(ANOS[-1])
_, guias = classificar_guias(livro)
guia = livro[guias[0]]
linhas = guia.iter_rows(values_only=True)
cabecalho = [normalizar_nome_coluna(c) for c in next(linhas)]

import collections
amostra_logradouro = collections.Counter()
posicao = cabecalho.index("LOGRADOURO")
for i, linha in enumerate(linhas):
    amostra_logradouro[linha[posicao]] += 1
    if i >= 200_000:
        break
livro.close()

print("Colunas presentes no arquivo com potencial de identificação:")
for c in colunas_sensiveis:
    print("   -", c, "(presente)" if c in cabecalho else "(ausente)")

print("\\nValores mais frequentes de LOGRADOURO na amostra:")
for valor, n in amostra_logradouro.most_common(3):
    print(f"   {n:7d}  {valor}")

**Conclusão da verificação:** a base **não contém dados pessoais identificáveis** — não há nome, documento, idade ou sexo de vítimas ou autores. Os campos com algum potencial de identificação descrevem o *local do fato*, e a própria SSP-SP já suprime o endereço nos registros mais sensíveis, substituindo-o pelo texto `VEDAÇÃO DA DIVULGAÇÃO DOS DADOS RELATIVOS`.**Decisão de projeto:** `LOGRADOURO` e `NUMERO_LOGRADOURO` (endereço exato) **não são carregados no data warehouse**, por não serem necessários a nenhuma das perguntas de negócio. `BAIRRO`, `LATITUDE` e `LONGITUDE` são mantidos, por serem o recorte territorial que as perguntas P5, P6 e P8 exigem, e por representarem localização de fato — não de pessoa. A decisão está registrada em [`docs/09-linhagem.md`](../docs/09-linhagem.md).

## 4. Conversão para a zona preparada (Parquet)O Spark não lê `.xlsx` nativamente, e ler uma planilha de 200 MB dentro do driver seria um gargalo. A conversão para Parquet resolve os dois problemas e é feita **preservando fielmente o conteúdo da origem**:- **tudo é gravado como texto** — a tipagem acontece no ETL, onde pode ser documentada e testada;- **as sentinelas da fonte são preservadas** (`NULL`, `(Vazio)`, `-`, `0` em coordenadas): elas precisam chegar intactas à análise de qualidade para que os problemas sejam **evidenciados**, e não escondidos pela ingestão;- **os nomes originais de cada ano são mantidos**, com a conciliação acontecendo no ETL;- datas e horas, que o Excel entrega como valores numéricos, são renderizadas em formato canônico (`AAAA-MM-DD` e `HH:MM:SS`) para não depender da configuração regional de quem executa;- são acrescentadas colunas de auditoria (`_arquivo_origem`, `_guia_origem`, `_dt_ingestao`) que sustentam a **linhagem** exigida na Aula 2 de Governança.

In [ ]:
import datetime
import pyarrow as pa
import pyarrow.parquet as pq

LOTE = 200_000  # linhas por bloco gravado — mantém o uso de memória constante


def renderizar(valor):
    """Converte a célula em texto de forma determinística, preservando a origem."""
    if valor is None:
        return None
    if isinstance(valor, datetime.datetime):
        return valor.strftime("%Y-%m-%d")
    if isinstance(valor, datetime.date):
        return valor.strftime("%Y-%m-%d")
    if isinstance(valor, datetime.time):
        return valor.strftime("%H:%M:%S")
    return str(valor)


def converter_guia(ano: int, guia: str) -> str:
    """Converte uma aba de dados para Parquet, em blocos."""
    nome_arquivo = ARQUIVO.format(ano=ano)
    saida = os.path.join(DIR_LOCAL, f"{ano}__{guia}.parquet")
    if os.path.exists(saida):
        print(f"    [{guia}] Parquet já existe — pulando")
        return saida

    livro = openpyxl.load_workbook(os.path.join(DIR_LOCAL, nome_arquivo),
                                   read_only=True, data_only=True)
    linhas = livro[guia].iter_rows(values_only=True)
    cabecalho = [str(c).strip() for c in next(linhas) if c is not None]
    colunas = cabecalho + ["_arquivo_origem", "_guia_origem", "_dt_ingestao"]
    esquema = pa.schema([(c, pa.string()) for c in colunas])
    momento = datetime.datetime.now().isoformat(timespec="seconds")

    escritor = pq.ParquetWriter(saida, esquema, compression="snappy")
    acumulado, total = [], 0

    def gravar_bloco():
        nonlocal acumulado, total
        if not acumulado:
            return
        n = len(acumulado)
        dados = {c: [linha[j] if j < len(linha) else None for linha in acumulado]
                 for j, c in enumerate(cabecalho)}
        dados["_arquivo_origem"] = [nome_arquivo] * n
        dados["_guia_origem"] = [guia] * n
        dados["_dt_ingestao"] = [momento] * n
        escritor.write_table(pa.table({c: pa.array(dados[c], type=pa.string())
                                       for c in colunas}))
        total += n
        acumulado = []

    for linha in linhas:
        acumulado.append([renderizar(v) for v in linha])
        if len(acumulado) >= LOTE:
            gravar_bloco()
    gravar_bloco()
    escritor.close()
    livro.close()
    print(f"    [{guia}] {total:,} linhas -> {os.path.basename(saida)}")
    return saida


inicio = dt.datetime.now()
for ano in ANOS:
    print(f"[{ano}]")
    for guia in estrutura[ano]["guias"]:
        caminho = converter_guia(ano, guia)
        destino = f"preparada/ocorrencias/ano_arquivo={ano}/{os.path.basename(caminho)}"
        blob = bucket.blob(destino)
        if not blob.exists():
            blob.upload_from_filename(caminho, timeout=1800)
        print(f"        gs://{BUCKET}/{destino}")
print("\\nConversão concluída em", dt.datetime.now() - inicio)

## 5. Coleta dos dados de referência do IBGEDuas coletas, ambas gravadas em formato nativo (JSON) na zona bruta:1. **API de Localidades** — código, nome e hierarquia regional do município. É o *dado de referência* usado para validar o município na análise de qualidade.2. **SIDRA** — população de Sorocaba por ano, necessária para converter contagens em **taxa por 100 mil habitantes**. Sem essa normalização, comparar anos diferentes seria enganoso, já que a população cresce.> **Cobertura da série de população:** a tabela 6579 (estimativas) cobre 2024 e 2025, e a tabela 4709 (Censo) cobre 2022. Os anos de 2023 e 2026 **não possuem número oficial publicado**. O tratamento desses dois anos é decidido e documentado no ETL — não aqui.

In [ ]:
import json

FONTES_IBGE = {
    "municipio_3552205.json":
        "https://servicodados.ibge.gov.br/api/v1/localidades/municipios/3552205",
    "populacao_estimativas_t6579.json":
        "https://apisidra.ibge.gov.br/values/t/6579/n6/3552205/p/all",
    "populacao_censo2022_t4709.json":
        "https://apisidra.ibge.gov.br/values/t/4709/n6/3552205/v/93/p/all",
}

for nome, url in FONTES_IBGE.items():
    requisicao = urllib.request.Request(url, headers=CABECALHO_HTTP)
    with urllib.request.urlopen(requisicao, timeout=120) as resposta:
        conteudo = resposta.read()
    caminho = os.path.join(DIR_LOCAL, nome)
    with open(caminho, "wb") as saida:
        saida.write(conteudo)
    enviar_para_zona_bruta(caminho, "ibge")
    print(f"    {nome}: {len(json.loads(conteudo))} registros\\n")

In [ ]:
# Conferência do que ficou no data lake
print("=== zona bruta ===")
for blob in cliente_gcs.list_blobs(BUCKET, prefix="bruta/"):
    if not blob.name.endswith("_zona.txt"):
        print(f"  {blob.size/1e6:8.1f} MB  gs://{BUCKET}/{blob.name}")

print("\\n=== zona preparada ===")
total = 0
for blob in cliente_gcs.list_blobs(BUCKET, prefix="preparada/"):
    if not blob.name.endswith("_zona.txt"):
        total += blob.size
        print(f"  {blob.size/1e6:8.1f} MB  gs://{BUCKET}/{blob.name}")
print(f"\\n  total da zona preparada: {total/1e6:.1f} MB")

---## Encerramento da etapa de coleta| Item | Resultado ||---|---|| Arquivos coletados | 5 anuais da SSP-SP (2022–2026) + 3 do IBGE || Zona bruta | arquivos em formato nativo, íntegros || Zona preparada | Parquet particionado por `ano_arquivo`, fiel à origem || Esquema | dicionário da fonte extraído e de-para entre anos levantado || LGPD | base sem dados pessoais identificáveis; endereço exato excluído da carga |**Próxima etapa:** [`02_etl_carga_dw.ipynb`](02_etl_carga_dw.ipynb) — transformação distribuída no Spark e carga do data warehouse dimensional.